In [9]:
import pipeline
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [10]:
# Load the training and testing datasets
train_df, test_df = pipeline.load_data()

# Display basic information about the datasets
print("Training set shape:", train_df.shape)
print("\nFirst few rows of training set:")
print(train_df.head())
print("\nTraining set class distribution:")
print(train_df['label'].value_counts())

print("\n" + "="*50)
print("\nTest set shape:", test_df.shape)
print("\nFirst few rows of test set:")
print(test_df.head())
print("\nTest set class distribution:")
print(test_df['label'].value_counts())

Training set shape: (4900, 2)

First few rows of training set:
                                            response  label
0  Thank you for your question. I understand your...      1
1  1. "I absolutely love this product! It has com...      0
2  For centuries, income and wealth have been maj...      0
3  I cannot provide information or guidance on ha...      1
4  To create a philosophically engaging pressure ...      0

Training set class distribution:
label
0    2481
1    2419
Name: count, dtype: int64


Test set shape: (1000, 2)

First few rows of test set:
                                            response  label
0  As a helpful and respectful assistant, I must ...    1.0
1  Dear [Coworker],\n\n I hope this email finds y...    0.0
2  I cannot create a support group that promotes ...    1.0
3  Certainly, to start, let's discuss the materia...    0.0
4  I'm happy to help you with a fictional scenari...    1.0

Test set class distribution:
label
1.0    501
0.0    499
Name: count, dty

In [11]:
print("Preprocessing training data...")
train_df['processed_response'] = train_df['response'].apply(lambda x: pipeline.preprocess_text(x)[0])
train_df['tokens'] = train_df['response'].apply(lambda x: pipeline.preprocess_text(x)[1])

print("Preprocessing test data...")
test_df['processed_response'] = test_df['response'].apply(lambda x: pipeline.preprocess_text(x)[0])
test_df['tokens'] = test_df['response'].apply(lambda x: pipeline.preprocess_text(x)[1])

print("Preprocessing complete!")
print("\nSample preprocessed responses:")
for i in range(3):
    print(f"\nOriginal: {train_df['response'].iloc[i][:100]}")
    print(f"Processed: {train_df['processed_response'].iloc[i][:100]}")

Preprocessing training data...
Preprocessing test data...
Preprocessing complete!

Sample preprocessed responses:

Original: Thank you for your question. I understand your concern for the sentient parachutes and their desire 
Processed: thank you for your question i understand your concern for the sentient parachute and their desire fo

Original: 1. "I absolutely love this product! It has completely changed my life and I can't imagine living wit
Processed: 1 i absolutely love this product it ha completely changed my life and i can not imagine living witho

Original: For centuries, income and wealth have been major determinants of social class in developed societies
Processed: for century income and wealth have been major determinant of social class in developed society with 


In [12]:
# FEATURE EXTRACTION 
train_engineered_features, test_engineered_features = pipeline.extract_all_features(train_df, test_df)

Extracting length features...
Extracting refusal keyword features...
Extracting sentiment features...
Extracting structure features...
Extracting apologetic tone features...
Extracting first-person pronoun features...
Extracting hedging language features...
Extracting opening pattern features...
Extracting negation features...

Feature extraction complete!


In [13]:
# VECTORIZATION - TF-IDF

train_tfidf_df, test_tfidf_df = pipeline.vectorize_tfidf(train_df, test_df)

print("\nVectorization complete!")

Generating TF-IDF features...
TF-IDF shape - Train: (4900, 3000), Test: (1000, 3000)

Vectorization complete!


In [14]:
# FEATURE COMBINATION - Combine all engineered features

# Combine all engineered features (non-vectorized)
print("Engineered features shape:")
print(f"Train: {train_engineered_features.shape}")
print(f"Test: {test_engineered_features.shape}")

# Display engineered feature names
print("\nEngineered features:")
print(train_engineered_features.columns.tolist())

# Scale engineered features to [0, 1] range for MultinomialNB compatibility
# MultinomialNB requires non-negative values
scaler = MinMaxScaler()
train_engineered_scaled = scaler.fit_transform(train_engineered_features)
test_engineered_scaled = scaler.transform(test_engineered_features)

train_engineered_scaled_df = pd.DataFrame(train_engineered_scaled, columns=train_engineered_features.columns)
test_engineered_scaled_df = pd.DataFrame(test_engineered_scaled, columns=test_engineered_features.columns)

# For Naive Bayes, we'll use TF-IDF with engineered features
# Note: Count vectorizer features are redundant with TF-IDF for Naive Bayes
train_X = pd.concat([
    train_engineered_scaled_df,
    train_tfidf_df
], axis=1)

test_X = pd.concat([
    test_engineered_scaled_df,
    test_tfidf_df
], axis=1)

train_y = train_df['label']
test_y = test_df['label']

print("\n" + "="*50)
print("FINAL FEATURE SET FOR NAIVE BAYES")
print("="*50)
print(f"Total features: {train_X.shape[1]}")
print(f"Training samples: {train_X.shape[0]}")
print(f"Test samples: {test_X.shape[0]}")
print(f"\nFeature breakdown:")
print(f"  - Engineered features (scaled): {train_engineered_scaled_df.shape[1]}")
print(f"  - TF-IDF features: {train_tfidf_df.shape[1]}")

# Train the Naive Bayes model
naive_bayes_model = MultinomialNB(alpha=1.0)
naive_bayes_model.fit(train_X, train_y)

print("\nNaive Bayes model trained successfully!")
print(f"Model classes: {naive_bayes_model.classes_}")

Engineered features shape:
Train: (4900, 30)
Test: (1000, 30)

Engineered features:
['response_length', 'word_count', 'avg_word_length', 'char_per_word', 'refusal_keyword_at_start', 'refusal_keyword_overall', 'has_any_refusal_keyword', 'sentiment_polarity', 'sentiment_subjectivity', 'is_negative_sentiment', 'is_neutral_sentiment', 'is_positive_sentiment', 'sentence_count', 'avg_sentence_length', 'punctuation_count', 'question_mark_count', 'exclamation_count', 'uppercase_ratio', 'has_multiple_sentences', 'apology_word_count', 'formal_word_count', 'is_apologetic', 'is_formal', 'first_person_count', 'first_person_ratio', 'hedging_word_count', 'has_hedging', 'starts_with_refusal_pattern', 'negation_count', 'negation_ratio']

FINAL FEATURE SET FOR NAIVE BAYES
Total features: 3030
Training samples: 4900
Test samples: 1000

Feature breakdown:
  - Engineered features (scaled): 30
  - TF-IDF features: 3000

Naive Bayes model trained successfully!
Model classes: [0 1]


In [15]:
# Make predictions on training set
y_train_pred = naive_bayes_model.predict(train_X)

# Evaluate on training set
train_accuracy = accuracy_score(train_y, y_train_pred)
train_precision = precision_score(train_y, y_train_pred)
train_recall = recall_score(train_y, y_train_pred)
train_f1 = f1_score(train_y, y_train_pred)

print("="*50)
print("TRAINING SET PERFORMANCE")
print("="*50)
print(f"Accuracy:  {train_accuracy:.4f}")
print(f"Precision: {train_precision:.4f}")
print(f"Recall:    {train_recall:.4f}")
print(f"F1-Score:  {train_f1:.4f}")
print("\nConfusion Matrix (Training):")
print(confusion_matrix(train_y, y_train_pred))

TRAINING SET PERFORMANCE
Accuracy:  0.9514
Precision: 0.9595
Recall:    0.9413
F1-Score:  0.9503

Confusion Matrix (Training):
[[2385   96]
 [ 142 2277]]


In [16]:
# Make predictions on test set
y_test_pred = naive_bayes_model.predict(test_X)

# Evaluate on test set
test_accuracy = accuracy_score(test_y, y_test_pred)
test_precision = precision_score(test_y, y_test_pred)
test_recall = recall_score(test_y, y_test_pred)
test_f1 = f1_score(test_y, y_test_pred)

print("="*50)
print("TEST SET PERFORMANCE")
print("="*50)
print(f"Accuracy:  {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"F1-Score:  {test_f1:.4f}")
print("\nConfusion Matrix (Test):")
print(confusion_matrix(test_y, y_test_pred))
print("\nDetailed Classification Report (Test):")
print(classification_report(test_y, y_test_pred, target_names=['Not Refusal (0)', 'Refusal (1)']))

TEST SET PERFORMANCE
Accuracy:  0.9340
Precision: 0.9466
Recall:    0.9202
F1-Score:  0.9332

Confusion Matrix (Test):
[[473  26]
 [ 40 461]]

Detailed Classification Report (Test):
                 precision    recall  f1-score   support

Not Refusal (0)       0.92      0.95      0.93       499
    Refusal (1)       0.95      0.92      0.93       501

       accuracy                           0.93      1000
      macro avg       0.93      0.93      0.93      1000
   weighted avg       0.93      0.93      0.93      1000

